In [0]:
select * from delta_catalog.demo_schema.customer_delta_bronze_2

In [0]:
%python
# demo_schema.customer_delta_bronze_2 --> is bronze table and we will write this to silver table --> demo_schema.customer_delta_silver_2

In [0]:
USE CATALOG delta_catalog;


In [0]:
CREATE TABLE demo_schema.customer_delta_silver_2
USING DELTA
LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/DemoCopyinto/schema/customer_delta_silver_2/'
AS
SELECT 
  CAST(Customerid AS INT) AS Customerid,
  Surname,
  CAST(Creditscore AS INT) AS Creditscore,
  Geography,
  Gender,
  CAST(Age AS INT) AS Age,
  CAST(Tenure AS INT) AS Tenure,
  CAST(Balance AS DOUBLE) AS Balance,
  CAST(Estimatedsalary AS DOUBLE) AS Estimatedsalary
FROM read_files(
  'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn/bankchurn_1_50.csv',
  format => 'csv',
  header => 'true'
);

In [0]:
drop table if exists delta_catalog.demo_schema.customer_delta_silver_2

In [0]:
%python
'''
Here since there is duplicate data in bronze table, we are using this window fucntion row_numbr to remove the duplicates. you can also run window fucntion seperately before.
'''

In [0]:
MERGE INTO delta_catalog.demo_schema.customer_delta_silver_2 target
USING (
    SELECT Customerid, Surname, Creditscore, Geography, Gender, Age, Tenure, Balance, Estimatedsalary
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY Customerid ORDER BY Customerid) AS rn
        FROM delta_catalog.demo_schema.customer_delta_bronze_2
    )
    WHERE rn = 1
) source
ON target.Customerid = source.Customerid
WHEN MATCHED THEN
            UPDATE
            SET
                target.Creditscore = source.Creditscore,
                target.Geography  = source.Geography,
                target.Gender = source.Gender,
                target.Age =source.Age,
                target.Tenure = source.Tenure,
                target.Balance = source.Balance,
                target.Estimatedsalary = source.Estimatedsalary

WHEN NOT MATCHED THEN 
INSERT *

In [0]:
describe history delta_catalog.demo_schema.customer_delta_silver_2

In [0]:
select * from delta_catalog.demo_schema.customer_delta_silver_2 timestamp as of '2026-06-08T17:02:39.000+00:00' except select * from delta_catalog.demo_schema.customer_delta_silver_2 timestamp as of '2026-06-08T16:51:47.000+00:00'


In [0]:
select distinct * from delta_catalog.demo_schema.customer_delta_silver_2

In [0]:
MERGE INTO delta.`/tmp/delta/delta_merge` AS target
USING _sqldf AS source
ON target.id = source.id
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED